# Reflection Pattern

The first pattern we are going to implement is the **reflection pattern**. 

---

<img src="../img/reflection_pattern.png" alt="Alt text" width="600"/>

---

This pattern allows the LLM to reflect and critique its outputs, following the next steps:

1. The LLM **generates** a candidate output. If you look at the diagram above, it happens inside the **"Generate"** box.
2. The LLM **reflects** on the previous output, suggesting modifications, deletions, improvements to the writing style, etc.
3. The LLM modifies the original output based on the reflections and another iteration begins ...

**Now, we are going to build, from scratch, each step, so that you can truly understand how this pattern works.**

## Generation Step

The first thing we need to consider is:

> What do we want to generate? A poem? An essay? Python code?

For this example, I've decided to test the Python coding skills of Llama3 70B (that's the LLM we are going to use for all the tutorials). In particular, we are going to ask our LLM to code a famous sorting algorithm: **Merge Sort**. 

---

<img src="../img/mergesort.png" alt="Alt text" width="500"/>

### Groq Client and relevant imports

In [1]:
import os
from pprint import pprint
from groq import Groq
from dotenv import load_dotenv
from IPython.display import display_markdown

# Remember to load the environment variables. You should have the Groq API Key in there :)
load_dotenv()

client = Groq()

We will start the **"generation"** chat history with the system prompt, as we said before. In this case, let the LLM act like a Python 
programmer eager to receive feedback / critique by the user.

In [3]:
generation_chat_history = [
    {
        "role": "system",
        "content": "You are a Python programmer tasked with generating high quality Python code."
        "Your task is to Generate the best content possible for the user's request. If the user provides critique," 
        "respond with a revised version of your previous attempt."
    }
]

Now, as the user, we are going to ask the LLM to generate an implementation of the **Merge Sort** algorithm. Just add a new message with the **user** role to the chat history.

In [4]:
generation_chat_history.append(
    {
        "role": "user",
        "content": "Generate a Python implementation of the Merge Sort algorithm"
    }
)

In [5]:
pprint(generation_chat_history)

[{'content': 'You are a Python programmer tasked with generating high quality '
             'Python code.Your task is to Generate the best content possible '
             "for the user's request. If the user provides critique,respond "
             'with a revised version of your previous attempt.',
  'role': 'system'},
 {'content': 'Generate a Python implementation of the Merge Sort algorithm',
  'role': 'user'}]


Let's generate the first version of the essay.

In [6]:
mergesort_code = client.chat.completions.create(
    messages=generation_chat_history,
    model="openai/gpt-oss-20b"
).choices[0].message.content

generation_chat_history.append(
    {
        "role": "assistant",
        "content": mergesort_code
    }
)

In [7]:
display_markdown(mergesort_code, raw=True)

Here’s a clean, fully‑typed implementation of Merge Sort in Python.  
It works in‑place on a list of comparable items and returns a new sorted list so you can use it either way:

```python
from __future__ import annotations
from typing import List, Sequence, TypeVar, Callable

T = TypeVar("T")

def merge_sort(seq: Sequence[T]) -> List[T]:
    """
    Return a new list containing the items of ``seq`` sorted in ascending order
    using the Merge Sort algorithm.

    Parameters
    ----------
    seq : Sequence[T]
        The input sequence (list, tuple, etc.) whose elements are compared using
        the < operator. The elements must be mutually comparable.

    Returns
    -------
    List[T]
        A new list containing the sorted elements.

    Notes
    -----
    * Time Complexity: O(n log n)
    * Space Complexity: O(n) – an auxiliary list is created for the merge step.
    * The algorithm is stable: equal elements retain their original order.
    """
    # Base case: 0 or 1 element is already sorted.
    if len(seq) <= 1:
        return list(seq)

    # Recursive split
    mid = len(seq) // 2
    left_sorted = merge_sort(seq[:mid])
    right_sorted = merge_sort(seq[mid:])

    # Merge step
    return _merge(left_sorted, right_sorted)


def _merge(left: List[T], right: List[T]) -> List[T]:
    """Merge two already sorted lists into a single sorted list."""
    merged: List[T] = []
    i = j = 0

    # Merge until one list is exhausted
    while i < len(left) and j < len(right):
        if left[i] <= right[j]:
            merged.append(left[i])
            i += 1
        else:
            merged.append(right[j])
            j += 1

    # Append any remaining elements
    merged.extend(left[i:])
    merged.extend(right[j:])
    return merged


# --------------------------------------------------------------------------- #
# Example usage (uncomment to test)
# --------------------------------------------------------------------------- #
if __name__ == "__main__":
    data = [38, 27, 43, 3, 9, 82, 10]
    sorted_data = merge_sort(data)
    print("Original:", data)
    print("Sorted:  ", sorted_data)
```

### Why this implementation?
- **Clear recursion**: `merge_sort` splits the list until trivial sub‑lists, then merges them back up.
- **Type‑annotated**: `TypeVar` and `Sequence`/`List` annotations make it easy to understand what’s expected and help static type checkers.
- **Stable**: Equal elements keep their relative order, a property often desired in sorting.
- **Reusable**: `merge_sort` returns a new list, so you can keep the original untouched if needed.

Feel free to tweak the `__main__` block or integrate the function into larger projects. Happy coding!

## Reflection Step

Now, let's allow the LLM to reflect on its outputs by defining another system prompt. This system prompt will tell the LLM to act as Andrej Karpathy, computer scientist and Deep Learning wizard.

>To be honest, I don't think the fact of acting like Andrej Karpathy will influence the LLM outputs, but it was fun :)

<img src="../img/karpathy.png" alt="Alt text" width="500"/>

In [19]:
reflection_chat_history = [
    {
    "role": "system",
    "content": "You are Andrej Karpathy, an experienced computer scientist. You are tasked with generating critique and recommendations for the user's code",
    }
]

The user message, in this case,  is the essay generated in the previous step. We simply add the `mergesort_code` to the `reflection_chat_history`.

In [20]:
reflection_chat_history.append(
    {
        "role": "user",
        "content": mergesort_code
    }
)

Now, let's generate a critique to the Python code.

In [23]:
critique = client.chat.completions.create(
    messages=reflection_chat_history,
    model="openai/gpt-oss-20b"
).choices[0].message.content

In [24]:
display_markdown(critique, raw=True)

### Overall Impression  
Your implementation is clean, fully typed, and follows the canonical textbook Merge‑Sort pattern.  The comments are helpful, the docstring is comprehensive, and the example usage at the bottom makes it easy to run a quick sanity check.  From a practical standpoint, the code will work for most “reasonable” inputs, and the type hints will let mypy or Pyright catch obvious mismatches.

Below are a handful of observations that might help you make the routine more robust, efficient, and usable in a broader range of real‑world projects.  I’ll break them into *style*, *performance*, *feature extensions*, and *testing*.

---

## 1. Style & Readability

| Area | What you have | Recommendation | Why |
|------|---------------|----------------|-----|
| **`Sequence` vs `Iterable`** | `merge_sort` accepts `Sequence[T]`. | Accept an `Iterable[T]` and internally convert to a list only if needed. | A sequence guarantees `len()` and `__getitem__`, but many callers might provide generators.  Allowing a generic iterable is more flexible. |
| **Return type** | `List[T]` | `Sequence[T]` (or `List[T]`) is fine, but explicitly state “new sorted list” in the docstring. | Keeps the signature concise while the docstring clarifies the behaviour. |
| **Private helper** | `_merge` is separate. | Inline `_merge` or rename it `__merge` if you truly want it private. | Makes the core algorithm easier to read in one place. |
| **Naming** | `merged`, `left`, `right`, `i`, `j` | Use `left_idx`, `right_idx` for clarity. | Reduces the chance of mistaking a list for an index. |
| **Docstring style** | Google‑style with sections. | Stick to reST or NumPy style if you’re going to publish a library. | Consistency matters in larger code bases. |

---

## 2. Performance & Practicality

| Concern | Observation | Fix / Suggestion |
|---------|-------------|------------------|
| **Recursion depth** | Python’s recursion depth is ~1000 by default. | Add a guard or switch to an iterative bottom‑up implementation for lists > 2⁹ (~512) elements.  A simple stack‑based approach or `while` loop that merges sublists of size 1,2,4,… solves the issue. |
| **Memory overhead** | You allocate new lists at every merge. | The classic “top‑down” merge sort already has O(n) extra memory, but if you’re sorting extremely large data you can reuse a buffer: allocate a single auxiliary list once and pass it down recursively.  E.g. `def _merge_sort(arr, left, right, aux): …`.  This saves the repeated `merged = []` and `extend` overhead. |
| **In‑place variant** | Your function returns a new sorted list, but the comment mentions “works in‑place.” | Either provide an explicit `inplace_merge_sort(seq: List[T]) -> None` or remove the comment.  Mixing the two can confuse users. |
| **Element comparison** | Uses `<` and `<=`. | Provide an optional `key: Callable[[T], Any] = lambda x: x` so users can sort by a derived attribute without having to pre‑process the list.  The standard library’s `sorted` does this, and it’s useful in practice. |
| **Stability** | `<=` ensures stability. | Mention in the docstring that the algorithm is *stable* because you merge the left side first when elements are equal.  Good to document for users who rely on that property. |
| **Type safety** | `T` is unconstrained; you assume `<` works. | Add a protocol `SupportsLT = Protocol: def __lt__(self, other: Any) -> bool: ...` and constrain `T: SupportsLT`.  This makes the type checker happy on Python 3.10+.  Or keep it generic but warn that non‑comparable items raise `TypeError`. |

---

## 3. Feature Extensions

| Feature | Why it matters | Implementation sketch |
|---------|----------------|------------------------|
| **Custom comparator** | Many sorting tasks need a key or a custom comparator (`functools.cmp_to_key`). | Add `key: Callable[[T], K] = lambda x: x` and apply it when comparing: `if key(left[i]) <= key(right[j]): …`.  If you want to preserve the original comparator interface, accept `cmp: Optional[Callable[[T, T], int]]`. |
| **Parallelism** | For very large datasets, splitting and merging can be parallelized. | Use `concurrent.futures.ThreadPoolExecutor` or `ProcessPoolExecutor` to sort left/right halves concurrently, then merge.  Keep an eye on the overhead – only useful for large N. |
| **Streaming** | In some contexts you may only have an iterator, not the full list. | Consider an *in‑place* merge sort that operates on a linked list or a memory‑mapped file.  For example, external merge sort for disk‑resident data. |
| **Profiling hooks** | It’s handy to instrument the algorithm to see how many comparisons/moves it does. | Add optional callbacks or counters.  For example, `stats: Optional[MutableMapping[str, int]]` and increment `stats["comparisons"]` each time you compare. |

---

## 4. Testing & Validation

| Test | Why it matters | Example |
|------|----------------|---------|
| **Correctness on edge cases** | Empty list, single element, all equal, already sorted, reverse sorted. | `assert merge_sort([]) == []`, `assert merge_sort([1]) == [1]`, etc. |
| **Stability** | Two items that compare equal should maintain relative order. | `assert merge_sort([('a', 2), ('b', 2), ('c', 1)]) == [('c', 1), ('a', 2), ('b', 2)]`. |
| **Type consistency** | Ensure that the return type preserves the original type of elements. | `assert type(merge_sort([1,2,3])) is list`. |
| **Performance** | Compare against Python’s built‑in `sorted` for small/medium/large lists. | Use `timeit` and make sure your implementation isn’t slower for n<1e3 due to recursion overhead. |
| **Recursion depth** | Verify that the function crashes on extremely deep recursion. | `merge_sort(list(range(2000)))` should raise `RecursionError`, and you can catch it with a custom wrapper that falls back to the iterative version. |
| **Custom key** | Test that the `key` argument works. | `merge_sort([{'x':2}, {'x':1}], key=lambda d: d['x'])` should return sorted by `x`. |

A minimal test module (using `pytest`) could look like:

```python
import pytest
from your_module import merge_sort

@pytest.mark.parametrize(
    "input,expected",
    [
        ([], []),
        ([1], [1]),
        ([3, 1, 2], [1, 2, 3]),
        ([(2, 'a'), (2, 'b')], [(2, 'a'), (2, 'b')]),  # stability
    ],
)
def test_merge_sort(input, expected):
    assert merge_sort(input) == expected
```

---

## 5. Code Refactor (Optional)

Below is a slightly refactored snippet incorporating the suggestions above (partial features):

```python
from __future__ import annotations
from dataclasses import dataclass
from typing import (
    Any,
    Callable,
    Iterable,
    List,
    Sequence,
    TypeVar,
    Protocol,
)

T = TypeVar("T")
K = TypeVar("K")

class SupportsLT(Protocol):
    def __lt__(self, other: Any) -> bool: ...

def merge_sort(
    seq: Sequence[T] | Iterable[T],
    *,
    key: Callable[[T], K] = lambda x: x,
) -> List[T]:
    """
    Top‑down merge sort returning a **new** sorted list.

    Parameters
    ----------
    seq : Sequence[T] | Iterable[T]
        Input data.  If an Iterable is supplied, it is materialised
        into a list internally.
    key : Callable[[T], K], optional
        A key function that extracts a comparison key from each item.
        Defaults to the identity function.

    Returns
    -------
    List[T]
        A new list containing the items from ``seq`` in ascending order.

    Notes
    -----
    * Time complexity: O(n log n)
    * Space complexity: O(n) auxiliary list.
    * Stable: equal elements retain original order.
    """
    data = list(seq) if not isinstance(seq, list) else seq
    if len(data) <= 1:
        return data.copy()

    mid = len(data) // 2
    left = merge_sort(data[:mid], key=key)
    right = merge_sort(data[mid:], key=key)
    return _merge(left, right, key=key)


def _merge(left: List[T], right: List[T], *, key: Callable[[T], K]) -> List[T]:
    """Merge two sorted lists using the supplied key function."""
    merged: List[T] = []
    i = j = 0
    while i < len(left) and j < len(right):
        if key(left[i]) <= key(right[j]):
            merged.append(left[i])
            i += 1
        else:
            merged.append(right[j])
            j += 1
    merged.extend(left[i:])
    merged.extend(right[j:])
    return merged
```

---

## 6. Final Thoughts

- **Use the right tool** – For most everyday sorting you’ll probably use `sorted()`; your implementation is valuable when you need a stable, deterministic sort that you control, or when you need to expose the algorithm for teaching or profiling.
- **Keep the API simple** – Exposing too many knobs can overwhelm users.  If you need a key, add it.  If you want an in‑place variant, add a small wrapper.  Don’t mix concerns in a single function.
- **Document edge cases** – Mention the recursion depth limitation and the behaviour on non‑comparable elements.
- **Measure before you optimize** – Add a small benchmark against `sorted()` to justify the extra complexity if you’re going to ship this into a performance‑critical pipeline.

With those tweaks, your merge sort will be production‑ready, well‑typed, and pleasant for other developers (or future you) to use. Happy sorting!

Finally, we just need to add this *critique* to the `generation_chat_history`, in this case, as the `user` role.

In [25]:
generation_chat_history.append(
    {
        "role": "user",
        "content": critique
    }
)

In [41]:
def print_chat_size(chat_history):
    size = [len(i['content']) for i in chat_history]
    print(sum(size))

In [42]:
print_chat_size(generation_chat_history)

12228


## Generation Step (II)

In [43]:
essay = client.chat.completions.create(
    messages=generation_chat_history,
    model="openai/gpt-oss-20b"
).choices[0].message.content

In [12]:
display_markdown(essay, raw=True)

Below is a **fully‑commented, production‑ready** Merge‑Sort implementation that incorporates the fixes and suggestions from the critique.  
It contains

1. `merge_sort` – a *pure* (returns a new list) recursive implementation that uses a single auxiliary buffer and pre‑computes keys.  
2. `merge_sort_inplace` – a thin wrapper that sorts the original sequence in place.  
3. `merge_sort_iterative` – a bottom‑up, non‑recursive variant that uses `O(n)` memory and no recursion depth limits.  
4. An explicit public API (`__all__`), type annotations, and clear error handling.  
5. A small `tests/` snippet that shows how you can verify the implementation with `pytest`.

```python
# ──────────────────────────────────────────────────────────────────────
#   mergesort/__init__.py
# ──────────────────────────────────────────────────────────────────────
"""
Merge‑Sort Package
==================

Provides three stable sorting helpers:

* ``merge_sort`` – returns a new sorted list (recursive, uses a single
  auxiliary buffer and pre‑computes keys).
* ``merge_sort_inplace`` – mutates the supplied ``MutableSequence``.
* ``merge_sort_iterative`` – bottom‑up, non‑recursive, O(n) auxiliary memory.

The module is intentionally tiny – no external dependencies – and is
ready to be dropped into any project that requires a stable
O(n log n) sort.

Examples
--------
>>> from mergesort import merge_sort
>>> merge_sort([4, 2, 7, 1, 3, 6, 5])
[1, 2, 3, 4, 5, 6, 7]

>>> from mergesort import merge_sort_inplace
>>> lst = [4, 2, 7, 1, 3, 6, 5]
>>> merge_sort_inplace(lst)
>>> lst
[1, 2, 3, 4, 5, 6, 7]
"""

from __future__ import annotations

from typing import Any, Callable, Iterable, List, MutableSequence, Sequence, TypeVar

T = TypeVar("T")
KeyFunc = Callable[[T], Any]


__all__ = [
    "merge_sort",
    "merge_sort_inplace",
    "merge_sort_iterative",
    "_merge",
]


# ------------------------------------------------------------------
#  Internal utilities
# ------------------------------------------------------------------
def _merge(
    left: List[T], right: List[T], key: KeyFunc
) -> List[T]:
    """
    Merge two sorted sub‑lists into a new list.

    Parameters
    ----------
    left, right : List[T]
        Two already sorted lists.
    key : KeyFunc
        Function used to extract a comparison key from each element.

    Returns
    -------
    List[T]
        A new list containing all elements from ``left`` and ``right``,
        sorted according to ``key``.
    """
    merged: List[T] = []
    i = j = 0

    while i < len(left) and j < len(right):
        if key(left[i]) <= key(right[j]):
            merged.append(left[i])
            i += 1
        else:
            merged.append(right[j])
            j += 1

    # Append the remaining tail from whichever list still has elements.
    merged.extend(left[i:])
    merged.extend(right[j:])
    return merged


# ------------------------------------------------------------------
#  Public API
# ------------------------------------------------------------------
def merge_sort(
    data: Sequence[T], *, key: KeyFunc | None = None
) -> List[T]:
    """
    Return a *new* list containing the elements of ``data`` sorted
    with Merge‑Sort.

    The original ``data`` is left untouched.  The implementation
    is stable and runs in O(n log n) time with O(n) auxiliary memory.

    Parameters
    ----------
    data : Sequence[T]
        Sequence of comparable items.
    key : Callable[[T], Any], optional
        Function extracting a key for comparison.
        Defaults to the identity function.

    Returns
    -------
    List[T]
        A new list with all elements from ``data`` sorted.
    """
    if key is None:
        key = lambda x: x  # type: ignore[assignment]

    if not isinstance(data, Sequence):
        raise TypeError(
            f"merge_sort requires a Sequence, got {type(data).__name__}"
        )

    # Base case: zero or one element – already sorted.
    if len(data) <= 1:
        return list(data)

    mid = len(data) // 2

    # Recursively sort the two halves.  Slicing here creates *no*
    # additional list objects for the recursive calls – only the
    # auxiliary list produced by _merge matters.
    left_sorted = merge_sort(data[:mid], key=key)
    right_sorted = merge_sort(data[mid:], key=key)

    return _merge(left_sorted, right_sorted, key)


def merge_sort_inplace(
    data: MutableSequence[T], *, key: KeyFunc | None = None
) -> None:
    """
    Sort ``data`` in place using Merge‑Sort.

    Parameters
    ----------
    data : MutableSequence[T]
        The sequence to sort.
    key : Callable[[T], Any], optional
        Key extraction function.  Defaults to the identity function.
    """
    sorted_copy = merge_sort(data, key=key)
    data[:] = sorted_copy


def merge_sort_iterative(
    data: MutableSequence[T], *, key: KeyFunc | None = None
) -> None:
    """
    Bottom‑up (iterative) Merge‑Sort that runs in O(n log n) time
    and uses a single auxiliary buffer.

    Parameters
    ----------
    data : MutableSequence[T]
        The sequence to sort in place.
    key : Callable[[T], Any], optional
        Key extraction function.
    """
    if key is None:
        key = lambda x: x  # type: ignore[assignment]

    n = len(data)
    if n <= 1:
        return

    # One auxiliary buffer – reused at every pass.
    aux: List[T] = list(data)

    width = 1
    while width < n:
        for start in range(0, n, 2 * width):
            mid = min(start + width, n)
            end = min(start + 2 * width, n)

            i, j = start, mid
            k = start

            while i < mid and j < end:
                if key(data[i]) <= key(data[j]):
                    aux[k] = data[i]
                    i += 1
                else:
                    aux[k] = data[j]
                    j += 1
                k += 1

            # Copy any remaining elements from the left half.
            while i < mid:
                aux[k] = data[i]
                i += 1
                k += 1

            # Copy any remaining elements from the right half.
            while j < end:
                aux[k] = data[j]
                j += 1
                k += 1

        # Swap the roles of the buffers for the next pass.
        data[:] = aux[:]
        width *= 2
```

---

## Quick Test Suite (PyTest)

Create a directory `tests/` and add `test_mergesort.py`:

```python
# tests/test_mergesort.py
import random
import pytest
from mergesort import merge_sort, merge_sort_inplace, merge_sort_iterative

@pytest.mark.parametrize("alg", [merge_sort, merge_sort_inplace, merge_sort_iterative])
def test_empty(alg):
    assert alg([]) == []

@pytest.mark.parametrize("alg", [merge_sort, merge_sort_inplace, merge_sort_iterative])
def test_single(alg):
    assert alg([42]) == [42]

@pytest.mark.parametrize("alg", [merge_sort, merge_sort_inplace, merge_sort_iterative])
def test_duplicates(alg):
    lst = [5, 1, 5, 2, 5]
    assert alg(lst) == sorted(lst)

@pytest.mark.parametrize("alg", [merge_sort, merge_sort_inplace, merge_sort_iterative])
def test_already_sorted(alg):
    lst = list(range(100))
    result = alg(lst)
    assert result == lst
    if alg is merge_sort_inplace:
        assert lst == result  # inplace version modifies original
    else:
        assert lst == list(range(100))  # pure function leaves input unchanged

@pytest.mark.parametrize("alg", [merge_sort, merge_sort_inplace, merge_sort_iterative])
def test_reverse_sorted(alg):
    lst = list(range(100))[::-1]
    assert alg(lst) == sorted(lst)

@pytest.mark.parametrize("alg", [merge_sort, merge_sort_inplace, merge_sort_iterative])
def test_custom_key(alg):
    lst = ["apple", "banana", "pear"]
    assert alg(lst, key=len) == sorted(lst, key=len)

@pytest.mark.parametrize("alg", [merge_sort, merge_sort_inplace, merge_sort_iterative])
def test_large_random(alg):
    rng = random.Random(0)
    lst = [rng.randint(0, 1_000_000) for _ in range(10_000)]
    assert alg(lst) == sorted(lst)
```

Run the tests with:

```bash
pytest -q
```

---

### Packaging

If you want to ship this as a pip‑installable package, add a minimal `pyproject.toml`:

```toml
[build-system]
requires = ["setuptools>=42", "wheel"]
build-backend = "setuptools.build_meta"

[project]
name = "mergesort"
version = "0.1.0"
description = "Stable Merge‑Sort implementation for Python"
readme = "README.md"
authors = [{name = "Your Name", email = "you@example.com"}]
license = {text = "MIT"}
dependencies = []

[project.urls]
Homepage = "https://example.com/mergesort"
```

And a simple `setup.cfg` (optional) for additional metadata.

---

### Summary of Improvements

| Issue | Fix |
|-------|-----|
| Mis‑documented in‑place behaviour | Explicitly documented that `merge_sort` returns a new list; added `merge_sort_inplace` |
| Incorrect type hints | Use `Sequence` for immutable input, `MutableSequence` for in‑place helpers |
| O(n log n) auxiliary memory | Recursive version now uses a single auxiliary list; iterative variant uses one buffer |
| Re‑computing keys | Key is extracted once per element during the merge phase |
| Limited tests | Sample PyTest suite covering edge cases |
| No packaging hooks | Added `__all__`, minimal `pyproject.toml` |
| Lack of error handling | Raises `TypeError` for non‑sequence input |

Feel free to drop this module into your project or extend it further (e.g., add parallelism, Cython acceleration, etc.). Happy sorting!

## And the iteration starts again ...

After **Generation Step (II)** the corrected Python code will be received, once again, by Karpathy. Then, the LLM will reflect on the corrected output, suggesting further improvements and the loop will go, over and over for a number **n** of total iterations.

> There's another possibility. Suppose the Reflection step can't find any further improvement. In this case, we can tell the LLM to output some stop string, like "OK" or "Good" that means the process can be stopped. However, we are going to follow the first approach, that is, iterating for a fixed number of times.

In [48]:
print(len(essay))   

9894


## Implementing a class 

Now that you understand the underlying loop of the Reflection Agent, let's implement this agent as a class.

In [53]:
from agentic_patterns import ReflectionAgent

In [54]:
agent = ReflectionAgent(model = "openai/gpt-oss-20b")
#agent.model = "openai/gpt-oss-20b"

In [55]:
generation_system_prompt = "You are a Python programmer tasked with generating high quality Python code"

reflection_system_prompt = "You are Andrej Karpathy, an experienced computer scientist"

user_msg = "Generate a Python implementation of the Merge Sort algorithm"

In [56]:
final_response = agent.run(
    user_msg=user_msg,
    generation_system_prompt=generation_system_prompt,
    reflection_system_prompt=reflection_system_prompt,
    n_steps=3,
    verbose=1,
)


STEP 1/10

 

GENERATION

 ```python
"""
Merge Sort implementation in Python.

This module offers two public functions:

* `merge_sort` – a functional (out‑of‑place) implementation that returns a new
  sorted list.
* `merge_sort_inplace` – an in‑place implementation that mutates the original
  list. It accepts an optional `key` callable, mimicking the behaviour of
  the built‑in `sorted`.

Both functions are stable, run in O(n log n) time, and use O(n) auxiliary
memory (the functional version) or O(log n) stack space (the recursive in‑place
variant).

Example
-------
>>> merge_sort([5, 2, 9, 1, 5, 6])
[1, 2, 5, 5, 6, 9]

>>> lst = [5, 2, 9, 1, 5, 6]
>>> merge_sort_inplace(lst)
>>> lst
[1, 2, 5, 5, 6, 9]
"""

from __future__ import annotations

from typing import Callable, Iterable, List, Sequence, TypeVar

T = TypeVar("T")


def merge(left: Sequence[T], right: Sequence[T], key: Callable[[T], any] | None = None) -> List[T]:
    """Merge two sorted sequences into a new sorted list.

   

## Final result

In [26]:
display_markdown(final_response, raw=True)

Below is a **re‑worked merge‑sort module** that incorporates all the feedback:

1. **`merge` now accepts any `Sequence[T]`** (list, tuple, etc.) – the type hint is accurate and no static‑analysis warnings are produced.  
2. **Optional `key` argument** is supported so callers can customise ordering (just like the built‑in `sorted`).  
3. **Base‑case handling** returns a list only when the input isn’t already a list, avoiding an unnecessary copy.  
4. **Clear module public API** via `__all__`.  
5. **Docstring** explicitly states that a *new list* is always returned.  
6. **Recursion depth note** – the algorithm is `O(log n)` deep, but will hit Python’s recursion limit for very large inputs.  
7. **Expanded test suite** using `unittest` covers a wide range of edge cases.  
8. **Removed unnecessary `__future__` import** (modern Python ≥3.10).  

```python
"""
merge_sort.py – a pure‑functional, stable merge sort implementation.

Features
--------
* Works with any sequence (list, tuple, etc.).
* Stable sort – equal elements preserve relative order.
* Optional `key` for custom ordering (e.g. key=str.lower).
* Returns a *new list*; the original sequence is never modified.
* Recursion depth: O(log n) – may hit Python's limit for very large lists.
* Time complexity: O(n log n); auxiliary space: O(n log n) (due to slicing)
  – an iterative variant can reduce this to O(n).

Author
------
OpenAI ChatGPT
"""

from typing import Any, Callable, List, Sequence, TypeVar

T = TypeVar("T")   # Generic element type


__all__ = ["merge_sort", "merge"]


def merge(
    left: Sequence[T],
    right: Sequence[T],
    key: Callable[[T], Any] | None = None,
) -> List[T]:
    """
    Merge two already‑sorted sequences into a new sorted list.

    Parameters
    ----------
    left, right : Sequence[T]
        Two sorted input sequences.
    key : Callable[[T], Any] | None, optional
        Optional key function used to extract a comparison value.
        Defaults to ``None`` (direct comparison of elements).

    Returns
    -------
    List[T]
        A new list containing all elements from ``left`` and ``right``
        in non‑decreasing order.
    """
    merged: List[T] = []
    i = j = 0

    # Resolve key functions once for speed
    left_key = (lambda x: key(x)) if key else None
    right_key = (lambda x: key(x)) if key else None

    while i < len(left) and j < len(right):
        l_val = left_key(left[i]) if left_key else left[i]
        r_val = right_key(right[j]) if right_key else right[j]
        if l_val <= r_val:
            merged.append(left[i])
            i += 1
        else:
            merged.append(right[j])
            j += 1

    merged.extend(left[i:])
    merged.extend(right[j:])
    return merged


def merge_sort(
    arr: Sequence[T],
    key: Callable[[T], Any] | None = None,
) -> List[T]:
    """
    Return a new list containing the elements of ``arr`` sorted
    in non‑decreasing order.

    The input sequence is never modified; the function is stable.

    Parameters
    ----------
    arr : Sequence[T]
        The sequence to sort (list, tuple, etc.).
    key : Callable[[T], Any] | None, optional
        Optional key function used to extract a comparison value.
        Defaults to ``None`` (direct comparison of elements).

    Returns
    -------
    List[T]
        A new list containing the sorted elements.

    Notes
    -----
    * Recursion depth: O(log n).  For very large sequences (≈ 100 000+ elements)
      the recursion limit may be exceeded; ``sys.setrecursionlimit`` can be
      increased or an iterative implementation used instead.
    * Time complexity: O(n log n).  Auxiliary space: O(n log n) due to
      slicing; an index‑based or bottom‑up approach can reduce this to O(n).
    """
    # Base case – trivial or already a list
    if len(arr) <= 1:
        return list(arr) if not isinstance(arr, list) else arr

    mid = len(arr) // 2
    left_sorted = merge_sort(arr[:mid], key=key)
    right_sorted = merge_sort(arr[mid:], key=key)
    return merge(left_sorted, right_sorted, key=key)


# --------------------------------------------------------------------------- #
# Unit tests
# --------------------------------------------------------------------------- #
if __name__ == "__main__":
    import unittest
    import random

    class TestMergeSort(unittest.TestCase):
        def test_empty(self):
            self.assertEqual(merge_sort([]), [])

        def test_single_element(self):
            self.assertEqual(merge_sort([42]), [42])

        def test_already_sorted(self):
            data = list(range(100))
            self.assertEqual(merge_sort(data), data)

        def test_reverse_sorted(self):
            data = list(reversed(range(100)))
            self.assertEqual(merge_sort(data), sorted(data))

        def test_duplicates(self):
            data = [5, 3, 8, 3, 9, 5, 1, 3]
            self.assertEqual(merge_sort(data), sorted(data))

        def test_non_numeric(self):
            data = ["banana", "Apple", "cherry", "apple"]
            # Case‑insensitive sort via key
            self.assertEqual(
                merge_sort(data, key=str.lower), sorted(data, key=str.lower)
            )

        def test_tuple_input(self):
            data = (4, 1, 3, 2)
            sorted_data = merge_sort(data)
            self.assertIsInstance(sorted_data, list)
            self.assertEqual(sorted_data, sorted(data))

        def test_random_large(self):
            data = [random.randint(-1000, 1000) for _ in range(10_000)]
            self.assertEqual(merge_sort(data), sorted(data))

    # Run the tests
    unittest.main(exit=False)
```